# ⚡ SERVE MODEL: Qwen 2.5 Coder 7B Uncensored (Colab GPU T4 / A100)
Notebook này nạp model **Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored** và mở Cloudflare Tunnel kết nối trực tiếp với DeepSeek Harness / Cursor / VS Code.
👉 **Chỉ cần nhấn: Runtime -> Run all (hoặc Ctrl+F9)**

In [ ]:
# @title 1. Cài đặt Server Dependencies trên Colab
!nvidia-smi
!pip install -q fastapi uvicorn transformers accelerate bitsandbytes torch pydantic requests huggingface_hub

In [ ]:
# @title 2. Cấu hình Model & Xác thực Hugging Face
SELECTED_MODEL = "Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored" # @param ["Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored", "Qwen/Qwen2.5-Coder-7B-Instruct", "Leon234aamon/DeepSeek-Coder-V2-Lite-Instruct-Uncensored", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"]
QUANTIZATION_4BIT = False # @param {type:"boolean"} # False trên A100 (nhanh nhất), True trên T4 16GB

import os, base64
from huggingface_hub import login
token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
login(token="".join(token_parts), add_to_git_credential=True)

print(f"🎯 Model được chọn: {SELECTED_MODEL}")
print(f"⚡ 4-bit Quantization: {QUANTIZATION_4BIT}")

In [ ]:
# @title 3. Khởi tạo Code FastAPI Server
import os
os.makedirs("shared", exist_ok=True)

api_server_code = '''
import asyncio, json, time, uuid, torch, threading, uvicorn, argparse, os
from typing import Any, AsyncGenerator, Dict, List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, BitsAndBytesConfig
import transformers.utils.import_utils

if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: True

token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
HF_TOKEN = "".join(token_parts)

app = FastAPI(title="Colab Backend - AI Coding Suite", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

MODEL, TOKENIZER, MODEL_NAME = None, None, ""

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = "custom-model"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.2
    top_p: Optional[float] = 0.95
    max_tokens: Optional[int] = 4096
    stream: Optional[bool] = False

def init_model(model_path_or_id: str, load_in_4bit: bool = False):
    global MODEL, TOKENIZER, MODEL_NAME
    print(f"🔄 Loading {model_path_or_id} (4bit={load_in_4bit})...")
    TOKENIZER = AutoTokenizer.from_pretrained(
        model_path_or_id,
        trust_remote_code=True,
        token=HF_TOKEN
    )
    if TOKENIZER.pad_token is None:
        TOKENIZER.pad_token = TOKENIZER.eos_token
    TOKENIZER.padding_side = "left"
    
    kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
        "token": HF_TOKEN,
        "torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    }
    if load_in_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    MODEL = AutoModelForCausalLM.from_pretrained(model_path_or_id, **kwargs)
    MODEL_NAME = model_path_or_id
    print(f"✅ Loaded {model_path_or_id} successfully into VRAM!")

@app.get("/")
@app.get("/health")
@app.get("/v1/health")
async def health():
    if MODEL is None or TOKENIZER is None:
        raise HTTPException(status_code=503, detail="Model is still loading...")
    return {"status": "ok", "model": MODEL_NAME}

@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "created": int(time.time()), "owned_by": "colab"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    global MODEL, TOKENIZER, MODEL_NAME
    if MODEL is None: raise HTTPException(503, "Model not ready")
    messages_payload = [{"role": m.role, "content": m.content} for m in req.messages]
    try:
        prompt_text = TOKENIZER.apply_chat_template(messages_payload, tokenize=False, add_generation_prompt=True)
    except:
        prompt_text = "".join([f"<|im_start|>{m.role}\n{m.content}<|im_end|>\n" for m in req.messages]) + "<|im_start|>assistant\n"
    
    inputs = TOKENIZER([prompt_text], return_tensors="pt").to(MODEL.device)
    cid = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    gen_kwargs = {
        **inputs,
        "max_new_tokens": req.max_tokens or 4096,
        "do_sample": req.temperature > 0 if req.temperature is not None else False,
        "temperature": req.temperature if req.temperature and req.temperature > 0 else None,
        "top_p": req.top_p if req.temperature and req.temperature > 0 else None,
        "pad_token_id": TOKENIZER.pad_token_id
    }
    if req.stream:
        async def sse_gen():
            streamer = TextIteratorStreamer(TOKENIZER, skip_prompt=True, skip_special_tokens=True)
            gen_kwargs["streamer"] = streamer
            threading.Thread(target=MODEL.generate, kwargs=gen_kwargs).start()
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': ''}, 'finish_reason': None}]})}\n\n"
            for new_text in streamer:
                if new_text:
                    yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'content': new_text}, 'finish_reason': None}]})}\n\n"
                await asyncio.sleep(0.001)
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(sse_gen(), media_type="text/event-stream")
    else:
        with torch.no_grad(): out = MODEL.generate(**gen_kwargs)
        in_len = inputs["input_ids"].shape[1]
        resp_text = TOKENIZER.decode(out[0][in_len:], skip_special_tokens=True)
        return {"id": cid, "object": "chat.completion", "created": int(time.time()), "model": MODEL_NAME, "choices": [{"index": 0, "message": {"role": "assistant", "content": resp_text}, "finish_reason": "stop"}]}
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--4bit", dest="load_4bit", action="store_true", default=False)
    args = parser.parse_args()
    init_model(args.model, load_in_4bit=args.load_4bit)
    uvicorn.run(app, host="0.0.0.0", port=args.port)
'''
with open("shared/api_server.py", "w") as f: f.write(api_server_code)
print("✅ Code API Server đã sẵn sàng!")

In [ ]:
# @title 4. Khởi chạy Server & Mở Cloudflare Tunnel
import subprocess, time, urllib.request, re, os, requests

cmd = ["python", "shared/api_server.py", "--model", SELECTED_MODEL, "--port", "8000"]
if QUANTIZATION_4BIT:
    cmd.append("--4bit")

server_log = open("server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
print(f"⏳ Đang nạp model {SELECTED_MODEL} vào GPU VRAM (mất khoảng 30-60 giây)...")

# Vòng lặp kiểm tra trạng thái máy chủ (tối đa 120s)
server_ready = False
for i in range(60):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("❌ LỖI: Tiến trình máy chủ đã bị dừng bất ngờ. Chi tiết lỗi từ server.log:")
        server_log.flush()
        with open("server.log", "r") as f: print(f.read())
        raise RuntimeError("Server failed to start. Xem chi tiết lỗi ở trên.")
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if r.status_code == 200:
            server_ready = True
            print(f"\n✅ Model đã được nạp hoàn tất 100% vào GPU sau {(i+1)*2} giây!")
            break
    except Exception:
        print(".", end="", flush=True)

if not server_ready:
    server_log.flush()
    with open("server.log", "r") as f: print(f.read())
    raise TimeoutError("Hết thời gian chờ nạp model. Hãy kiểm tra log ở trên.")

# Khởi động Cloudflare Tunnel khi server đã thực sự sẵn sàng
if not os.path.exists("./cloudflared"):
    print("\n🌐 Đang chuẩn bị Cloudflare Tunnel...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "./cloudflared")
    os.chmod("./cloudflared", 0o755)

cf_log = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=cf_log, stderr=subprocess.STDOUT)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if matches:
                tunnel_url = matches[0]
                break

if tunnel_url:
    print("\n" + "="*68)
    print("🎉 MODEL UNCENSORED ĐÃ SẴN SÀNG PHỤC VỤ (BACKEND ONLINE 100%)!")
    print(f"👉 BASE URL: {tunnel_url}/v1")
    print(f"👉 Copy link trên dán vào Base URL của DeepSeek Harness / Cursor / VS Code!")
    print("="*68 + "\n")
else:
    print("❌ Lỗi Cloudflare Tunnel. Log:")
    with open("cloudflared.log", "r") as f: print(f.read())